In [3]:
from pathlib import Path
OLD_BASE = "/mnt/aix23102/deepfake/FakeAVCeleb_new"   # what my JSONs currently reference
NEW_BASE = str(Path.home() / "SIR-25-26" / "FakeAVCeleb_v1.2")  # where my raw dataset lives
# write into referee/data/ 
print("OLD_BASE:", OLD_BASE)
print("NEW_BASE:", NEW_BASE)

OLD_BASE: /mnt/aix23102/deepfake/FakeAVCeleb_new
NEW_BASE: /home/jupyter-ashah2/SIR-25-26/FakeAVCeleb_v1.2


In [4]:
import json
import pandas as pd
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

def replace_base_in_path(p: str, old_base: str, new_base: str):
    if not isinstance(p, str):
        return p
    if p.startswith(old_base):
        return p.replace(old_base, new_base, 1)
    return p.replace(old_base, new_base)  # fallback replace

def fix_csv(csv_in, csv_out, old_base, new_base):
    df = pd.read_csv(csv_in)
    if "new_path" in df.columns:
        df["new_path"] = df["new_path"].astype(str).apply(lambda p: replace_base_in_path(p, old_base, new_base))
    df.to_csv(csv_out, index=False)
    print(f"Wrote fixed csv to {csv_out} (rows: {len(df)})")

def fix_json(json_in, json_out, old_base, new_base, keys_to_fix=("target_file","reference_file","file_path")):
    with open(json_in, "r") as f:
        data = json.load(f)
    def fix_item(item):
        if isinstance(item, dict):
            for k in keys_to_fix:
                if k in item and isinstance(item[k], str):
                    item[k] = replace_base_in_path(item[k], old_base, new_base)
        return item

    if isinstance(data, dict) and "data" in data:
        data["data"] = [fix_item(i) for i in data["data"]]
    elif isinstance(data, list):
        data = [fix_item(i) for i in data]
    else:
        raise RuntimeError("Unexpected JSON structure")

    with open(json_out, "w") as f:
        json.dump(data, f, indent=2)
    print(f"Wrote fixed json to {json_out} (entries: {len(data)})")


In [5]:
# Paths to the original files in referee/data 
orig_csv = Path("data/all_real_with_split.csv")
orig_train = Path("data/train_set.json")
orig_val = Path("data/val_set.json")
orig_test = Path("data/test_pairs.json")

if not orig_csv.exists():
    raise FileNotFoundError(f"Expected CSV at {orig_csv}. Adjust path or move CSV into referee/data/.")

# outputs 
fixed_csv = Path("data/all_real_with_split_fixed.csv")
fixed_train = Path("data/train_set_fixed.json")
fixed_val = Path("data/val_set_fixed.json")
fixed_test = Path("data/test_pairs_fixed.json")

# Run the fixes
fix_csv(str(orig_csv), str(fixed_csv), OLD_BASE, NEW_BASE)
for inp, out in [(orig_train, fixed_train), (orig_val, fixed_val), (orig_test, fixed_test)]:
    if inp.exists():
        fix_json(str(inp), str(out), OLD_BASE, NEW_BASE)
    else:
        print("Warning: not found", inp)


Wrote fixed csv to data/all_real_with_split_fixed.csv (rows: 37642)
Wrote fixed json to data/train_set_fixed.json (entries: 14003)
Wrote fixed json to data/val_set_fixed.json (entries: 1077)
Wrote fixed json to data/test_pairs_fixed.json (entries: 6464)


In [6]:
import os
for p in (fixed_csv, fixed_train, fixed_val, fixed_test):
    print(p, "exists?", p.exists(), "size:", (p.stat().st_size if p.exists() else "N/A"))


data/all_real_with_split_fixed.csv exists? True size: 8355797
data/train_set_fixed.json exists? True size: 3477108
data/val_set_fixed.json exists? True size: 265127
data/test_pairs_fixed.json exists? True size: 2348656
